In [ ]:
import numpy as np
import sklearn
import sklearn.preprocessing
import sklearn.linear_model
import matplotlib.pyplot as plt

from scipy.optimize import linprog


## Dataset

In [ ]:
ds1 = np.genfromtxt('../datasets/ds_altezzapeso.csv', delimiter=';')
ds2 = np.genfromtxt('../datasets/ds_altezzapeso2.csv', delimiter=';')

### Caso non separabile

In [ ]:
x_ns = np.block([[ds1[:, :]], [ds2[:, :]]])
y_ns = np.block([np.ones(len(ds1)), -1*np.ones(len(ds2))])
i = (y_ns==1)
j = (y_ns==-1)

plt.xlabel("Altezza (m)")
plt.ylabel("Peso (Kg)")
plt.plot(x_ns[i,0], x_ns[i,1], '.', x_ns[j,0], x_ns[j,1], '.')
plt.show()

x_ns = sklearn.preprocessing.minmax_scale(x_ns)

### Caso separabile

In [ ]:
i1 = ds1[:,1]<69
i2 = ds2[:,1]>71
x_s = np.block([[ds1[i1,:]], [ds2[i2,:]]])
y_s = np.block([np.ones(np.sum(i1)), -1*np.ones(np.sum(i2))])
i = (y_s==1)
j = (y_s==-1)

plt.xlabel("Altezza (m)")
plt.ylabel("Peso (Kg)")
plt.plot(x_s[i,0],x_s[i,1],'.',x_s[j,0],x_s[j,1],'.')
plt.show()

x_s = sklearn.preprocessing.minmax_scale(x_s) #NORMALIZZAZIONE


In [ ]:
import warnings

def perceptron(x, y, max_iter=10000, learning_rate=1):

    x = np.hstack([np.ones((len(x), 1)), x ])  # bias in posizione 0
    w = np.zeros(x.shape[1])

    for _ in range(max_iter):
        miss = np.where(y * (x @ w) <= 0)[0]          # punti classificati male
        if len(miss) == 0:                             # convergenza: dati separati
            return w

        k = miss[0]
        w = w + learning_rate * y[k] * x[k]

    warnings.warn(f"Nessuna convergenza in {max_iter} iterazioni "
                  f"(dati non linearmente separabili?).")
    return w


def linear_classifier(ds,y):
    ds = sklearn.preprocessing.add_dummy_feature(ds)
    y = np.asarray(y)
    n, d1 = ds.shape

    A = y[:, None] * ds
    res = linprog(
        c=np.zeros(d1),
        A_ub=-A,
        b_ub=-np.ones(n),
        bounds=(None, None),   # variabili libere
        method="highs",
    )
    print(res.message)
    if not res.success:
        raise RuntimeError("Problema non risolvibile: dati non linearmente separabili?")
    return res.x

## Regressione Logistica

### Caso separabile

In [ ]:
clf = sklearn.linear_model.LogisticRegression(C = 1000, max_iter=30000)
clf.fit(x_s,y_s)

print(clf.coef_[0]/clf.intercept_, clf.intercept_/clf.intercept_)

yt = clf.predict(x_s)
yp = clf.predict_proba(x_s)[:,1]

it = y_s==1
jt = y_s==-1

ysep = (-1*clf.coef_[0,0]*x_s[:,0] - clf.intercept_)/clf.coef_[0,1]

plt.figure()
plt.title("Logistic Regression")
plt.plot(x_s[it,0],x_s[it,1],'.')
plt.plot(x_s[jt,0],x_s[jt,1],'.')
plt.plot(x_s[:,0],ysep,'-')
# on_margin = np.array([np.abs(yp-0.5)<0.25])
# plt.plot(x_ns[on_margin[0],0],x_ns[on_margin[0],1],'ro')
plt.show()

### Caso non separabile

In [ ]:
clf = sklearn.linear_model.LogisticRegression(max_iter=3000)
clf.fit(x_ns,y_ns)

print(clf.coef_[0]/clf.intercept_, clf.intercept_/clf.intercept_)

yt = clf.predict(x_ns)
yp = clf.predict_proba(x_ns)[:,1]

it = y_ns==1
jt = y_ns==-1

ysep = (-1*clf.coef_[0,0]*x_ns[:,0] - clf.intercept_)/clf.coef_[0,1]

plt.figure()
plt.title("Logistic Regression")
plt.plot(x_ns[it,0],x_ns[it,1],'.')
plt.plot(x_ns[jt,0],x_ns[jt,1],'.')
plt.plot(x_ns[:,0],ysep,'-')
# on_margin = np.array([np.abs(yp-0.5)<0.25])
# plt.plot(x_ns[on_margin[0],0],x_ns[on_margin[0],1],'ro')
plt.ion()
plt.show()
plt.pause(1.0001)

## Classificatore Lineare

### Caso separabile

In [ ]:
w = linear_classifier(x_s, y_s)
print(w[1:]/w[0], w[0]/w[0])

it = y_s == 1
jt = y_s == -1
x1 = np.linspace(0, 1, 100)   # range dei dati normalizzati

eq = rf"${w[0]:.2f} {w[1]:+.2f}\,x_1 {w[2]:+.2f}\,x_2 = 0$"

# Forma 1: x2 in funzione di x1  ->  x2 = -(w1/w2) x1 - w0/w2
wp0 = -w[1]/w[2]
wp1 = -w[0]/w[2]
plt.plot(x_s[it,0], x_s[it,1], '.', label="Classe +1")
plt.plot(x_s[jt,0], x_s[jt,1], '.', label="Classe -1")
plt.plot(x1, wp0*x1 + wp1, '-', label=eq)
plt.title(eq)
plt.xlabel(r"$x_1$ (altezza normalizzata)")
plt.ylabel(r"$x_2$ (peso normalizzato)")
plt.xlim(0, 1); plt.ylim(0, 1)
plt.legend()
plt.show()

### Caso non separabile

In [ ]:
w = linear_classifier(x_ns,y_ns)
print(w[1:]/w[0], w[0]/w[0])

it = y_ns==1
jt = y_ns==-1

wp0 = -w[1]/w[2]
wp1 = -w[0]/w[2]
plt.plot(x_ns[it,0],x_ns[it,1],'.',x_ns[jt,0],x_ns[jt,1],'.',x_ns[:,0],wp0*x_ns[:,0]+wp1,'-')
plt.show()

wp0 = -w[2]/w[1]
wp1 = -w[0]/w[1]
plt.plot(x_ns[it,0],x_ns[it,1],'.',x_ns[jt,0],x_ns[jt,1],'.',x_ns[:,0],wp0*x_ns[:,0]+wp1,'-')
plt.show()


## Algoritmo del percettrone

### Caso separabile

In [ ]:
w = perceptron(x_s,y_s)
print(w[1:]/w[0], w[0]/w[0])

it = y_s==1
jt = y_s==-1

wp0 = -w[1]/w[2]
wp1 = -w[0]/w[2]

plt.title("Perceptron")
plt.plot(x_s[i,0],x_s[i,1],'.',x_s[j,0],x_s[j,1],'.',x_s[:,0],wp0*x_s[:,0]+wp1,'-')
plt.show()

### Caso non separabile

In [ ]:
w = perceptron(x_ns,y_ns)
print(w[1:]/w[0], w[0]/w[0])

it = y_ns==1
jt = y_ns==-1

wp0 = -w[1]/w[2]
wp1 = -w[0]/w[2]

plt.title("Perceptron")
plt.plot(x_ns[it,0],x_ns[it,1],'.',x_ns[jt,0],x_ns[jt,1],'.',x_ns[:,0],wp0*x_ns[:,0]+wp1,'-')
plt.show()